# DOAgent — Grid-world demo

This notebook runs the **grid-world** scenario step by step: multiple agents explore a grid with partial observations and share discovered cells via DOAgent. It only uses the `doagent` library and code defined here. Run in Google Colab.

**What you'll do:** Install doagent → define a minimal grid env and policies (heuristic **and** LLM-based) → create a session with file as the shared data model → run the loop (with shared map) → run analysis including causal attribution.

One of the four agents (`agent_3`) uses an **LLM-based policy**: it calls an actual LLM to reason about the best exploration action. If no API key is available, the notebook falls back to a heuristic policy so you can still run it end-to-end.

## Step 1 — Install the library

Install DOAgent from the repository. No PettingZoo needed for this demo.

In [ ]:
!pip install -q git+https://github.com/cabrerac/doagent.git openai

## Step 2 — Imports

Import Session, make_env, RunReporter, and the analysis modules.

In [ ]:
import json
import os
import random
from typing import Any, Dict, List, Tuple

from doagent import Session, RunReporter, make_env
from doagent.analysis import accountability, interpretability, provenance, traceability

## Step 3 — Define a minimal grid-world environment

Default params match **examples/gridworld_demo_config.yaml** (15×15, 4 agents, 2 landmarks, 100 max_cycles). The env must provide:

- `reset(seed=...)` → observations dict (per agent) with `position`, `cells`, `width`, `height`
- `step(actions)` → dict with `observations`, `rewards`, `terminations`
- `agents` → list of agent ids

Actions: 0=stay, 1=left, 2=right, 3=up, 4=down. Reward = number of newly discovered cells for that agent.

In [ ]:
class SimpleGridEnv:
    def __init__(self, width=4, height=4, agent_ids=None, landmarks=1, observation_radius=1, max_cycles=15, seed=None):
        self._w, self._h = width, height
        self._agent_ids = agent_ids or ["agent_0", "agent_1"]
        self._landmarks = landmarks
        self._radius = observation_radius
        self._max_cycles = max_cycles
        self._rng = random.Random(seed)
        self._positions = {}
        self._landmark_positions = []
        self._discovered = set()
        self._step_count = 0

    @property
    def agents(self):
        return list(self._agent_ids)

    def _rand_pos(self):
        return (self._rng.randrange(self._w), self._rng.randrange(self._h))

    def _observe(self, agent_id):
        x, y = self._positions[agent_id]
        cells = []
        for dx in range(-self._radius, self._radius + 1):
            for dy in range(-self._radius, self._radius + 1):
                cx, cy = x + dx, y + dy
                if 0 <= cx < self._w and 0 <= cy < self._h:
                    value = "landmark" if (cx, cy) in self._landmark_positions else "empty"
                    cells.append({"x": cx, "y": cy, "value": value})
        return {"position": {"x": x, "y": y}, "cells": cells, "width": self._w, "height": self._h}

    def _move(self, pos, action):
        x, y = pos
        if action == 1: x -= 1
        elif action == 2: x += 1
        elif action == 3: y += 1
        elif action == 4: y -= 1
        return (max(0, min(self._w - 1, x)), max(0, min(self._h - 1, y)))

    def reset(self, *, seed=None):
        if seed is not None:
            self._rng.seed(seed)
        self._step_count = 0
        self._positions = {aid: self._rand_pos() for aid in self._agent_ids}
        self._landmark_positions = [self._rand_pos() for _ in range(self._landmarks)]
        self._discovered.clear()
        obs = {aid: self._observe(aid) for aid in self._agent_ids}
        for o in obs.values():
            for c in o["cells"]:
                self._discovered.add((c["x"], c["y"]))
        return obs

    def step(self, actions):
        self._step_count += 1
        for aid, a in actions.items():
            if aid in self._positions:
                self._positions[aid] = self._move(self._positions[aid], a)
        obs = {aid: self._observe(aid) for aid in self._agent_ids}
        rewards = {}
        for aid, o in obs.items():
            new_cells = sum(1 for c in o["cells"] if (c["x"], c["y"]) not in self._discovered)
            for c in o["cells"]:
                self._discovered.add((c["x"], c["y"]))
            rewards[aid] = float(new_cells)
        done = self._step_count >= self._max_cycles
        return {"observations": obs, "rewards": rewards, "terminations": {aid: done for aid in self._agent_ids}}

def create_grid_env(*, width=15, height=15, agent_ids=None, landmarks=2, observation_radius=1, max_cycles=100, seed=None):
    return SimpleGridEnv(width=width, height=height, agent_ids=agent_ids or ["agent_0", "agent_1", "agent_2", "agent_3"],
                         landmarks=landmarks, observation_radius=observation_radius, max_cycles=max_cycles, seed=seed)

## Step 4 — Define policies and build_shared_map

Policies match **examples/gridworld_demo/policies.py**: `grid_random` (random walk toward unknown cells), `grid_frontier` (move toward nearest unknown), `grid_auction_frontier` (frontier with bid), and `grid_llm` (LLM-based exploration). `build_shared_map` turns recorded agent_update payloads into a merged map. This is the **shared data model** in action: agents coordinate through records; the library records contributions and the topology controls visibility.

The `grid_llm` policy sends the agent's observation and shared map context to an LLM, which reasons about the best exploration action. When the LLM is uncertain, it can **abstain** — returning `status: "abstain"`. The library captures the LLM call as a **reasoning trace** automatically via tool tracing.

In [ ]:
def _move_towards(src: Tuple[int, int], dst: Tuple[int, int]) -> int:
    dx, dy = dst[0] - src[0], dst[1] - src[1]
    if dx == 0 and dy == 0:
        return 0
    if abs(dx) >= abs(dy):
        return 2 if dx > 0 else 1
    return 3 if dy > 0 else 4

def _neighbors(pos: Tuple[int, int]):
    x, y = pos
    return [(x - 1, y, 1), (x + 1, y, 2), (x, y + 1, 3), (x, y - 1, 4)]

def _known_cells(shared_map: Dict) -> set:
    return {(c.get("x"), c.get("y")) for c in shared_map.get("cells", []) if c.get("x") is not None and c.get("y") is not None}

def _grid_bounds(observation: Dict) -> Tuple[int, int]:
    w, h = observation.get("width"), observation.get("height")
    return (int(w) if w is not None else 0, int(h) if h is not None else 0)

def grid_random(params: Dict):
    rng = random.Random(params.get("seed", 0))
    def decide(request):
        obs = request.get("inputs", {}).get("observation", {})
        shared_map = request.get("inputs", {}).get("shared_map", {})
        pos = obs.get("position", {})
        x, y = int(pos.get("x", 0)), int(pos.get("y", 0))
        width, height = _grid_bounds(obs)
        known = _known_cells(shared_map)
        unknown_actions = []
        valid_actions = []
        for nx, ny, action in _neighbors((x, y)):
            if width and height and (nx < 0 or ny < 0 or nx >= width or ny >= height):
                continue
            valid_actions.append(action)
            if (nx, ny) not in known:
                unknown_actions.append(action)
        action = rng.choice(unknown_actions) if unknown_actions else (rng.choice(valid_actions) if valid_actions else 0)
        return {"choice": {"status": "act", "action": action}}
    return decide

def grid_frontier(params: Dict):
    rng = random.Random(params.get("seed", 0))
    def decide(request):
        obs = request.get("inputs", {}).get("observation", {})
        shared_map = request.get("inputs", {}).get("shared_map", {})
        pos = obs.get("position", {})
        x, y = int(pos.get("x", 0)), int(pos.get("y", 0))
        width, height = _grid_bounds(obs)
        known = _known_cells(shared_map)
        if width == 0 or height == 0:
            return {"choice": {"status": "act", "action": 0}}
        unknown_cells = [(ux, uy) for ux in range(width) for uy in range(height) if (ux, uy) not in known]
        if not unknown_cells:
            return {"choice": {"status": "act", "action": 0}}
        nearest = min(unknown_cells, key=lambda c: abs(c[0] - x) + abs(c[1] - y))
        action = _move_towards((x, y), nearest)
        if action == 0:
            action = rng.choice([1, 2, 3, 4])
        return {"choice": {"status": "act", "action": action}}
    return decide

def grid_auction_frontier(params: Dict):
    rng = random.Random(params.get("seed", 0))
    def decide(request):
        obs = request.get("inputs", {}).get("observation", {})
        shared_map = request.get("inputs", {}).get("shared_map", {})
        pos = obs.get("position", {})
        x, y = int(pos.get("x", 0)), int(pos.get("y", 0))
        width, height = _grid_bounds(obs)
        known = _known_cells(shared_map)
        unknown_cells = [(ux, uy) for ux in range(width) for uy in range(height) if (ux, uy) not in known]
        if not unknown_cells:
            return {"choice": {"status": "act", "action": 0, "bid": 0.0}}
        nearest = min(unknown_cells, key=lambda c: abs(c[0] - x) + abs(c[1] - y))
        distance = abs(nearest[0] - x) + abs(nearest[1] - y)
        action = _move_towards((x, y), nearest)
        if action == 0:
            action = rng.choice([1, 2, 3, 4])
        bid = 1.0 / (distance + 1.0)
        return {"choice": {"status": "act", "action": action, "bid": bid}}
    return decide

def build_shared_map(records):
    cells = {}
    for r in records:
        payload = getattr(r, "payload", r) if not isinstance(r, dict) else r.get("payload", {})
        for c in (payload.get("local_knowledge", {}).get("observation", {}).get("cells", []) or payload.get("cells", [])):
            if c.get("x") is not None and c.get("y") is not None:
                cells[(c["x"], c["y"])] = c.get("value", "unknown")
    return {"cells": [{"x": x, "y": y, "value": v} for (x, y), v in cells.items()]}


# ---------- LLM-based policy ----------

_LLM_SYSTEM_PROMPT = (
    "You are an agent in a multi-agent environment. "
    "Given the current observation, choose the best action. "
    "Respond with a JSON object containing exactly these fields:\n"
    '  "action": <integer action id>,\n'
    '  "confidence": <float 0.0-1.0>,\n'
    '  "reasoning": "<brief explanation of your choice>"\n'
    "If you cannot determine a good action, set action to null and confidence to 0.0."
)

_GRIDWORLD_ACTION_SPACE = {0: "stay", 1: "left", 2: "right", 3: "up", 4: "down"}

def _build_gridworld_prompt(observation, action_space, goal):
    pos = observation.get("position", {})
    cells = observation.get("cells", [])
    w = observation.get("width", "?")
    h = observation.get("height", "?")
    shared_map = observation.get("shared_map", {})
    known_count = len(shared_map.get("cells", []))
    actions_desc = "\n".join(f"  {k}: {v}" for k, v in sorted(action_space.items()))
    visible = [f"({c.get('x')},{c.get('y')})" + (" (landmark)" if c.get("value") == "landmark" else "") for c in cells]
    return (
        f"Goal: {goal}\n\n"
        f"You are at position ({pos.get('x')}, {pos.get('y')}) on a {w}x{h} grid.\n"
        f"Visible cells: {', '.join(visible)}.\n"
        f"Known cells from shared map: {known_count}.\n\n"
        f"Available actions:\n{actions_desc}\n\n"
        "Choose the action that best helps discover unknown cells. "
        "Respond with JSON only."
    )

def _parse_llm_output(raw):
    if isinstance(raw, str):
        text = raw
    elif hasattr(raw, "choices"):
        msg = getattr(raw.choices[0], "message", raw.choices[0])
        text = getattr(msg, "content", None) or str(msg)
    elif isinstance(raw, dict):
        choices = raw.get("choices", [])
        text = choices[0].get("message", {}).get("content", "") if choices else json.dumps(raw)
    else:
        text = str(raw)
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
    return json.loads(text)

def grid_llm(params):
    model = params.get("model", "gemini-3.1-flash-lite-preview")
    threshold = float(params.get("confidence_threshold", 0.3))
    action_space = _GRIDWORLD_ACTION_SPACE
    build_prompt = params.get("build_prompt", _build_gridworld_prompt)
    system_prompt = params.get("system_prompt", _LLM_SYSTEM_PROMPT)

    def decide(request):
        tools = request.get("tools", {})
        llm = tools.get("llm")
        if llm is None:
            return {"choice": {"status": "error", "action": None, "error": "No 'llm' tool provided."}}

        obs = request.get("inputs", {}).get("observation", {})
        goal = request.get("goal", "discover unknown cells")
        user_prompt = build_prompt(obs, action_space, goal)

        try:
            raw = llm(model=model, messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ])
        except Exception as exc:
            return {"choice": {"status": "error", "action": None, "error": f"LLM call failed: {exc}"}}

        try:
            parsed = _parse_llm_output(raw)
        except (json.JSONDecodeError, KeyError, IndexError) as exc:
            return {"choice": {"status": "error", "action": None, "error": f"Parse error: {exc}"}}

        action = parsed.get("action")
        confidence = float(parsed.get("confidence", 0.0))
        reasoning_text = parsed.get("reasoning", "")

        if action is None or confidence < threshold:
            return {
                "choice": {"status": "abstain", "action": None},
                "reasoning": {"source": "llm", "text": reasoning_text, "confidence": confidence},
                "explanation": reasoning_text or "Low confidence — abstaining.",
            }

        if isinstance(action, (int, float)):
            action = int(action)
        return {
            "choice": {"status": "act", "action": action},
            "reasoning": {"source": "llm", "text": reasoning_text, "confidence": confidence},
            "explanation": reasoning_text,
        }
    return decide

## Step 5 — Configure session with file as the shared data model

The session is configured with **file as the shared data model** and matches **examples/gridworld_demo_config.yaml**. **Decentralisation:** we set `topology: peer_to_peer` with visibility so each agent sees only listed peers' records. **Openness:** we set `participation: True` so the session has a **participation registry** (`session.participation_registry`); in the run loop we call register/deregister when agents rejoin or leave (energy model), so the library knows who is participating.

**LLM setup:** `agent_3` uses the `grid_llm` policy backed by Google Gemini (free tier). If a Gemini API key is available (set `DOAGENT_GEMINI_API_KEY` as a Colab secret or env var), the agent calls the LLM for real. If the key is missing, we fall back to the `grid_random` heuristic so the notebook is still runnable. Get a free key at [Google AI Studio](https://aistudio.google.com/apikey).

In [ ]:
output_base = "./output"
agent_ids = ["agent_0", "agent_1", "agent_2", "agent_3"]

llm_tool = None
llm_available = False

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("DOAGENT_GEMINI_API_KEY")
except Exception:
    api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("DOAGENT_GEMINI_API_KEY")

if api_key:
    try:
        from google import genai
        from google.genai import types
        _client = genai.Client(api_key=api_key)

        def _gemini_call(*, model, messages):
            system_parts = [m["content"] for m in messages if m["role"] == "system"]
            user_parts = [m["content"] for m in messages if m["role"] != "system"]
            config = types.GenerateContentConfig(
                system_instruction="\n".join(system_parts) if system_parts else None,
            )
            resp = _client.models.generate_content(
                model=model, contents="\n".join(user_parts), config=config,
            )
            return resp.text or ""

        llm_tool = _gemini_call
        llm_available = True
        print("Gemini LLM available — agent_3 will use the grid_llm policy.")
    except ImportError:
        print("google-genai package not found — agent_3 will fall back to grid_random.")
else:
    print("No GEMINI_API_KEY found — agent_3 will fall back to grid_random.")

agent_3_config = (
    {
        "id": "agent_3",
        "policy": {"name": "grid_llm", "params": {"model": "gemini-3.1-flash-lite-preview", "confidence_threshold": 0.3}},
        "tools": {"llm": llm_tool},
        "metadata": {"explanation": "LLM-based exploration: uses an LLM to reason about the best action."},
    }
    if llm_available
    else {
        "id": "agent_3",
        "policy": {"name": "grid_random", "params": {"seed": 3}},
        "metadata": {"explanation": "Fallback: random exploration (LLM not available)."},
    }
)

configs = [
    {
        "id": "agent_0",
        "policy": {"name": "grid_frontier", "params": {}},
        "metadata": {"explanation": "Frontier policy: move toward nearest unknown cells."},
    },
    {
        "id": "agent_1",
        "policy": {"name": "grid_random", "params": {"seed": 1}},
        "metadata": {"explanation": "Random exploration with local unknown-cell preference."},
    },
    {
        "id": "agent_2",
        "policy": {"name": "grid_auction_frontier", "params": {"seed": 2}},
        "metadata": {"explanation": "Auction-frontier policy: prioritize frontier cells with highest bid."},
    },
    agent_3_config,
]
session = Session.from_config({
    "shared_data": {"type": "file"},
    "scenario_name": "gridworld",
    "output_base": output_base,
    "run_config": {"logging_level": 2},
    "topology": {"mode": "peer_to_peer", "visibility": {"agent_0": ["agent_1"], "agent_1": ["agent_2"], "agent_3": ["agent_0", "agent_1"]}},
    "participation": True,
    "policies": {"grid_random": grid_random, "grid_frontier": grid_frontier, "grid_auction_frontier": grid_auction_frontier, "grid_llm": grid_llm},
})
env = make_env(create_grid_env, width=15, height=15, agent_ids=agent_ids, landmarks=2, observation_radius=1, max_cycles=100, seed=999)
print(f"Run id: {session.run_id}")

## Step 6 — Wrap env, create agents, run the loop (energy model + openness)

Same run loop as **examples/gridworld_demo/gridworld_demo.py**: an **energy model**—agents are skipped when energy ≤ 0 and re-included when recharged above threshold. Only active agents decide and act.

**Openness:** The session has a **participation registry** (`session.participation_registry`). When an agent "leaves" (energy ≤ 0) we call `session.deregister_participant(agent_id)`; when they "rejoin" (recharged above threshold) we call `session.register_participant(agent_id, capabilities=["map_discovery"])`. So the library knows who is currently participating through Session-level APIs. Params: energy_min=8, energy_max=10, energy_decay=1, energy_recharge=1, energy_leave_threshold=2. No env rendering in the notebook; run the local example for a visual display.

In [ ]:
rounds = 100
seed = 999
energy_model = True
energy_min, energy_max = 8, 10
energy_decay, energy_recharge = 1, 1
energy_leave_threshold = 2

wrapped = session.wrap_env(env, env_actor="gridworld_env")
agents = session.create_agents(configs, goal="map_discovery", payload_type="map_update")
registry = session.participation_registry
if registry and energy_model:
    for aid in agent_ids:
        session.register_participant(aid, capabilities=["map_discovery"])
observations = wrapped.reset(seed=seed)
rng = random.Random(seed)
active_agents = set(agent_ids)
energy_levels = {aid: rng.randint(energy_min, energy_max) for aid in agent_ids}

for round_id in range(1, rounds + 1):
    if energy_model:
        for aid in list(active_agents):
            energy_levels[aid] -= energy_decay
            if energy_levels[aid] <= 0:
                active_agents.remove(aid)
                if registry:
                    session.deregister_participant(aid)
        for aid in agent_ids:
            if aid in active_agents:
                continue
            energy_levels[aid] = min(energy_levels[aid] + energy_recharge, energy_max)
            if energy_levels[aid] > energy_leave_threshold:
                active_agents.add(aid)
                if registry:
                    session.register_participant(aid, capabilities=["map_discovery"])
    active_ids = sorted(active_agents)
    actions = {}
    for aid in active_ids:
        shared_records = session.visible_records(aid, kind="agent_update")
        shared_map = build_shared_map(shared_records)
        result = agents[aid].decide(observations.get(aid, {}), round_id, inputs={"observation": observations.get(aid, {}), "shared_map": shared_map})
        actions[aid] = result["action"] if result["action"] is not None else 0
    step = wrapped.step(actions)
    observations = step["observations"]
print("Run completed.")

## Step 7 — Run analysis (provenance, traceability, accountability, interpretability)

Call each analysis module with `write_output=True`. The library writes PNG/PDF and JSON under `output/<run_id>/analysis/`. Grid-world has a discovery semantics so we include **accountability** (causal attribution). Use the effective id from provenance for interpretability.

In [ ]:
run_id = session.run_id
effective_id = provenance.render_chain_tree("last", run_id, output_base=output_base, write_output=True)
traceability.build_trace_graph(run_id, output_base=output_base, write_output=True)
accountability.causal_attribution(run_id, output_base=output_base, write_output=True)
last_id = effective_id or "last"
interpretability.build_atomic_explanations(last_id, run_id, output_base=output_base, write_output=True)
print(f"Analysis written to {output_base}/{run_id}/analysis/")
print("  - provenance/: provenance_tree.png, .pdf")
print("  - traceability/: trace_graph.png, .pdf")
print("  - accountability/: causal_attribution.png, .pdf")
print("  - interpretability/: atomic_explanations_for_last.json")

## Step 8 — View and interpret the analysis results

The analysis step wrote analysis artefacts under `output/<run_id>/analysis/`. Each block below shows one output and a short explanation.

These analysis tools are an expandable demonstration set of the analysis DOAgent enables; extend or adapt them for your scenario.

**Full guide (legends, layouts, filenames):** [interpreting-analysis.md](https://github.com/cabrerac/doagent/blob/main/guides/interpreting-analysis.md) — in a local clone: `guides/interpreting-analysis.md`.

#### Provenance tree

Chain of records that led to the last outcome—which decisions and env steps produced the final state.

**How to read it:** Node colors: outcome (light blue), agent_update (light green), trace (gold), initial_state (dark). Arrows: derived_from (blue), trace_to (red), enabled_by (green), from (orange).

In [ ]:
from pathlib import Path
from IPython.display import Image, display

base = Path(output_base) / run_id / "analysis"
provenance_png = base / "provenance" / "provenance_tree.png"
if provenance_png.exists():
    display(Image(filename=str(provenance_png)))
else:
    print("Provenance tree not found (re-run Step 7).")

#### Trace graph

Cause–effect links between records (who acted, what they observed, what changed).

**How to read it:** Nodes: initial state (dark), regular states (light blue), dedup convergence (gold). Edges colored by agent (agent_0 blue, agent_1 orange, etc.).

In [ ]:
trace_png = base / "traceability" / "trace_graph.png"
if trace_png.exists():
    display(Image(filename=str(trace_png)))
else:
    print("Trace graph not found (re-run Step 7).")

#### Causal attribution

For discovery-style runs: which agents discovered which cells, and how much each agent contributed (productive vs redundant decisions). Useful to attribute "who found what."

**How to read it:** Left: cumulative discovery per agent over rounds (line color = agent). Middle: total cells discovered per agent (bar color = agent). Right: productive (green) vs redundant (red) transitions per agent; percentage = effectiveness.

In [ ]:
attr_png = base / "accountability" / "causal_attribution.png"
if attr_png.exists():
    display(Image(filename=str(attr_png)))
else:
    print("Causal attribution figure not found (re-run Step 7).")

#### Atomic explanations

Transition-level explanation units for the chosen outcome (e.g. the last step).

Below we render a human-readable view first, then a small JSON sample.

**JSON fields (each unit):** `from_state_id`, `to_state_id`, `agent_id`, `decision_id`, `decision_action`, `level`, optional `rationale_text`, plus traceability links/evidence refs and `rendered_text`.

In [ ]:
import json
from collections import Counter

atomic_path = base / "interpretability" / "atomic_explanations_for_last.json"
if atomic_path.exists():
    with open(atomic_path, encoding="utf-8") as f:
        data = json.load(f)

    print(f"Interpretability: wrote analysis/interpretability/ ({len(data)} atomic explanation units)")
    levels = Counter(u.get("level", "?") for u in data)
    print(f"  Levels: {dict(levels)}")
    for idx, unit in enumerate(data[:8], start=1):
        print(f"  {idx:02d}. {unit.get('rendered_text', '(missing rendered_text)')}")

    print("\nJSON sample (first 2 units):")
    print(json.dumps(data[:2], indent=2, default=str))
else:
    print("Atomic explanations file not found (re-run Step 7).")

---
You can download the generated files from Colab (e.g. from the file browser) or inspect `output/<run_id>/analysis/` in your runtime.